# 🎲 Baseline splitters: uninformed and gold-standard controls

Welcome! The `baseline` family gathers five splitters that don't use chemistry at all — they
operate purely on record count and (optionally) a label array `y`. They exist as **controls**:
any "smarter", chemistry-aware splitter should be benchmarked against these, because a fancy
split that doesn't outperform `random` on the metric it's supposed to improve isn't earning its
complexity.

This notebook walks through all five classes — `RandomSplitter`, `StratifiedRandomSplitter`,
`KFoldSplitter`, `MonteCarloSplitter`, `PredefinedSplitter` — each with a short explanation of
what it does, its real trade-offs (straight from its docstring), and a runnable example.

<a id="1"></a>
## 1. 🎲 The `baseline` family

In [1]:
import numpy as np

from chemsplit.splitters.baseline import (
    KFoldSplitter,
    MonteCarloSplitter,
    PredefinedSplitter,
    RandomSplitter,
    StratifiedRandomSplitter,
)


def summarize(result):
    print(f"train={len(result.train)} valid={len(result.valid)} "
          f"test={len(result.test)} discard={len(result.discard)}")

X = np.arange(200, dtype=np.float64).reshape(-1, 1)
rng = np.random.default_rng(0)
y = (rng.random(200) < 0.2).astype(np.int64)  # imbalanced binary label, ~20% positive
print("X:", X.shape, "  y positives:", int(y.sum()), "/", len(y))

X: (200, 1)   y positives: 37 / 200


All five splitters below consume this shared 200-record dataset, so the effect of each
splitting strategy on the very same data is directly comparable.

<a id="1.1"></a>
### 1.1 🪙 `RandomSplitter` — the honest, chemistry-free control

A plain shuffled (or index-order) train/test split with no regard for labels or chemistry — the
cheapest possible split, and the correct control for *"is my pipeline wired correctly?"*. See the
Pitfalls below for why it is the wrong tool for *"will this generalise?"*.

| Parameter | Meaning |
|---|---|
| `train_size` | fraction of records assigned to train |
| `test_size` | fraction of records assigned to test |
| `random_state` | RNG seed (ignored when `shuffle=False`) |

> 💡 **Advantages**
> - Linear-time and dependency-free: no featurization, no chemistry, nothing to compute.
> - The only split that leaves train and test distributionally identical by construction, so it is the right control for "is my pipeline wired correctly?" rather than "will this generalise?".
> - An unbiased estimate of interpolation error within the dataset's own chemical space — the correct answer when the deployment library is drawn from that same space.
> - Lower variance across seeds than any structural split, making small model differences easier to detect.

> ⚠️ **Pitfalls**
> - Scatters congeneric series across train and test, so near-duplicates land on both sides and reported metrics overstate prospective performance, often by 0.2-0.4 in R² or ROC-AUC.
> - On datasets built from a handful of papers, a random split can be close to a memorisation test.
> - Says nothing about scaffold generalisation, temporal drift, or assay-protocol shift.
> - Reporting only a random-split number is the most common cause of irreproducible QSAR results.

In [2]:
sp = RandomSplitter(train_size=0.7, test_size=0.3, random_state=0)
[result] = sp.split_result(X)
summarize(result)

train=140 valid=0 test=60 discard=0


70/30 as requested, with no attempt to balance `y` — that's exactly what makes it the
right *and* the wrong tool, depending on the question being asked.

<a id="1.2"></a>
### 1.2 ⚖️ `StratifiedRandomSplitter` — random, but label-balanced

Random split stratified on the label (class balance for classification, quantile/uniform/k-means
bins for regression), sized per-stratum by largest-remainder apportionment so realised sizes are
exactly reproducible and off by at most one record per stratum.

| Parameter | Meaning |
|---|---|
| `train_size` | fraction of records assigned to train |
| `test_size` | fraction of records assigned to test |
| `random_state` | RNG seed |

> 💡 **Advantages**
> - Guarantees every class, or every label decile, appears in every partition — essential for imbalanced data where a naive random split can leave a fold with zero actives.
> - Cuts the variance of ROC-AUC/PR-AUC/R² estimates on small datasets, often more than any change to the model.
> - The same mechanism handles regression through quantile binning, so one splitter serves both task types.

> ⚠️ **Pitfalls**
> - Only controls the **label** distribution; says nothing about chemical similarity, and the word "stratified" tempts people to treat it as a rigorous split.
> - Stratifying on the label leaks the label distribution into the split design; on very small `n` this mildly biases the test set toward looking like train.
> - Multi-task stratification is genuinely hard — prefer `BalancedMultiTaskSplitter` for sparse multi-task matrices.

In [3]:
sp = StratifiedRandomSplitter(train_size=0.8, test_size=0.2, random_state=0)
[result] = sp.split_result(X, y=y)
summarize(result)
train_pos = float(np.mean(y[result.train]))
test_pos = float(np.mean(y[result.test]))
print(f"positive rate: train={train_pos:.3f} test={test_pos:.3f} (overall={y.mean():.3f})")

train=160 valid=0 test=40 discard=0
positive rate: train=0.188 test=0.175 (overall=0.185)


The positive rate in train and test tracks the overall ~20% rate almost exactly — the
apportionment guarantee at work.

`StratifiedRandomSplitter` isn't limited to binary classification: `task="regression"` bins a continuous label into `n_bins` quantile/uniform/k-means strata before splitting, and `multitask=` controls what happens when `y` has more than one column (the default, `multitask="error"`, refuses rather than silently stratifying on the wrong thing).

In [4]:
rng2 = np.random.default_rng(1)
y_reg = rng2.normal(loc=5.0, scale=2.0, size=200)  # a continuous label, e.g. pIC50

sp = StratifiedRandomSplitter(
    task="regression", binning="quantile", n_bins=5, train_size=0.8, test_size=0.2, random_state=0,
)
[result] = sp.split_result(X, y=y_reg)
summarize(result)
print(f"train mean={y_reg[result.train].mean():.3f}  test mean={y_reg[result.test].mean():.3f}  "
      f"overall mean={y_reg.mean():.3f}")

train=160 valid=0 test=40 discard=0
train mean=4.835  test mean=4.925  overall mean=4.853


Quantile binning keeps the *distribution* of the continuous label comparable across train and test, not just its mean — check `result.metadata["bin_edges"]` when a heavily tied label (e.g. a censored assay floor) produces degenerate bins.

In [5]:
y_multi = np.stack([y, (rng2.random(200) < 0.35).astype(np.int64)], axis=1)  # 2-task label matrix

sp = StratifiedRandomSplitter(
    multitask="sum_labels", train_size=0.8, test_size=0.2, random_state=0,
)
[result] = sp.split_result(X, y=y_multi)
summarize(result)
print("per-task positive rate, train:", y_multi[result.train].mean(axis=0).round(3))
print("per-task positive rate, test: ", y_multi[result.test].mean(axis=0).round(3))

train=161 valid=0 test=39 discard=0
per-task positive rate, train: [0.186 0.391]
per-task positive rate, test:  [0.179 0.385]


`multitask="sum_labels"` is a documented crude heuristic (it stratifies on the row-wise label sum), and `"iterative"` is only approximate — both are best-effort. For sparse multi-task matrices prefer `BalancedMultiTaskSplitter` (see the `similarity` notebook), which balances coverage per task directly instead of collapsing labels into one proxy.

<a id="1.3"></a>
### 1.3 🔁 `KFoldSplitter` — standard k-fold cross-validation

Standard (optionally stratified, optionally leave-one-out) k-fold cross-validation: returns one
`SplitResult` per fold, sharing a single permutation across all folds.

| Parameter | Meaning |
|---|---|
| `n_splits` | number of folds, or `"loo"` for leave-one-out |
| `shuffle` | whether to shuffle before folding |
| `random_state` | RNG seed |

> 💡 **Advantages**
> - Every record serves for both training and evaluation, cutting estimator variance on small assay-sized datasets (n < 2000).
> - Yields a *distribution* of scores instead of a single number, so model comparisons can be tested statistically rather than eyeballed.
> - Composes with any grouping — feed group labels from any group-forming splitter into `GroupKFoldSplitter` for the structural analogue.

> ⚠️ **Pitfalls**
> - K-fold is a **resampling protocol, not a split criterion**: plain `KFoldSplitter` inherits every weakness of `random`.
> - Fold scores aren't independent — training sets overlap by `(k-2)/(k-1)` — so a naive standard error across folds understates uncertainty.
> - Selecting hyperparameters on the same folds used for reporting inflates the score; use `NestedCVSplitter` instead.

In [6]:
sp = KFoldSplitter(n_splits=5, shuffle=True, random_state=0)
results = sp.split_result(X)
print(f"{len(results)} folds")
for i, r in enumerate(results):
    print(f"  fold {i}:", end=" ")
    summarize(r)

5 folds
  fold 0: train=160 valid=0 test=40 discard=0
  fold 1: train=160 valid=0 test=40 discard=0
  fold 2: train=160 valid=0 test=40 discard=0
  fold 3: train=160 valid=0 test=40 discard=0
  fold 4: train=160 valid=0 test=40 discard=0


Five folds, each holding out a disjoint ~20% of records as test — together the test sets
cover the whole dataset exactly once.

Both `KFoldSplitter` and `MonteCarloSplitter` accept `stratify=True` to keep each fold's/repeat's label balance close to the overall rate — the k-fold analogue of `StratifiedRandomSplitter`.

In [7]:
sp = KFoldSplitter(n_splits=5, shuffle=True, stratify=True, random_state=0)
results = sp.split_result(X, y=y)
for i, r in enumerate(results):
    print(f"  fold {i}: positive rate train={y[r.train].mean():.3f} test={y[r.test].mean():.3f}", end=" | ")
    summarize(r)

  fold 0: positive rate train=0.182 test=0.195 | train=159 valid=0 test=41 discard=0
  fold 1: positive rate train=0.182 test=0.195 | train=159 valid=0 test=41 discard=0
  fold 2: positive rate train=0.188 test=0.175 | train=160 valid=0 test=40 discard=0
  fold 3: positive rate train=0.186 test=0.179 | train=161 valid=0 test=39 discard=0
  fold 4: positive rate train=0.186 test=0.179 | train=161 valid=0 test=39 discard=0


<a id="1.4"></a>
### 1.4 🔀 `MonteCarloSplitter` — repeated shuffle-split resampling

Repeated independent random splits (`ShuffleSplit`); unlike `KFoldSplitter`, test sets across
repeats are **not** disjoint — each repeat is an independent draw, decoupling test-set size from
the repeat count.

| Parameter | Meaning |
|---|---|
| `n_splits` | number of repeats |
| `train_size` | fraction of records assigned to train, per repeat |
| `test_size` | fraction of records assigned to test, per repeat |
| `random_state` | RNG seed |

> 💡 **Advantages**
> - Decouples test-set size from the repeat count, so a 10% test set can be evaluated 50 times — not possible with k-fold.
> - The spread across repeats directly estimates split-induced variance, which for chemical data often exceeds the gap between the models being compared.

> ⚠️ **Pitfalls**
> - Test sets overlap across repeats, so results are correlated and the naive standard error is optimistic.
> - Some records may never land in any test set; the last fold's `metadata` reports coverage.
> - Inherits every chemical-leakage weakness of `random`.

In [8]:
sp = MonteCarloSplitter(n_splits=5, train_size=0.7, test_size=0.3, random_state=0)
results = sp.split_result(X)
print(f"{len(results)} repeats")
for i, r in enumerate(results):
    print(f"  repeat {i}:", end=" ")
    summarize(r)

5 repeats
  repeat 0: train=140 valid=0 test=60 discard=0
  repeat 1: train=140 valid=0 test=60 discard=0
  repeat 2: train=140 valid=0 test=60 discard=0
  repeat 3: train=140 valid=0 test=60 discard=0
  repeat 4: train=140 valid=0 test=60 discard=0


Sizes are identical across repeats (as requested), but — unlike k-fold — the actual test
*indices* differ each time.

In [9]:
sp = MonteCarloSplitter(n_splits=5, train_size=0.7, test_size=0.3, stratify=True, random_state=0)
results = sp.split_result(X, y=y)
for i, r in enumerate(results):
    print(f"  repeat {i}: positive rate train={y[r.train].mean():.3f} test={y[r.test].mean():.3f}", end=" | ")
    summarize(r)

  repeat 0: positive rate train=0.186 test=0.183 | train=140 valid=0 test=60 discard=0
  repeat 1: positive rate train=0.186 test=0.183 | train=140 valid=0 test=60 discard=0
  repeat 2: positive rate train=0.186 test=0.183 | train=140 valid=0 test=60 discard=0
  repeat 3: positive rate train=0.186 test=0.183 | train=140 valid=0 test=60 discard=0
  repeat 4: positive rate train=0.186 test=0.183 | train=140 valid=0 test=60 discard=0


Every repeat now tracks the overall ~20% positive rate instead of drifting with the random draw — useful when repeats are few or `y` is heavily imbalanced.

<a id="1.5"></a>
### 1.5 📌 `PredefinedSplitter` — wrap someone else's split exactly

Wraps an externally supplied train/valid/test/discard assignment instead of computing one — the
only way to reproduce a published benchmark's numbers exactly, and the correct way to replace an
internal (non-shareable) time split with a shareable index list.

| Parameter | Meaning |
|---|---|
| `assignment` | sequence of partition names, or a mapping of partition name → index list |
| `fold_column` | per-record fold id (`-1` = always train); yields one fold per distinct id |

> 💡 **Advantages**
> - The only way to reproduce a published benchmark's numbers exactly.
> - Zero ambiguity: the split is data, not an algorithm, so it can be shipped, diffed, and checksummed.
> - Lets an internal time split that can't be published be replaced with a shareable index list.

> ⚠️ **Pitfalls**
> - Comparability is the *only* guarantee — some widely used benchmark splits (including some MoleculeNet scaffold splits) contain near-duplicate leakage across the boundary, and inheriting the split inherits the flaw.
> - A published split is tied to a specific row order; filtering or re-standardising the dataset silently shifts what the indices point to. Always re-key on InChIKey and verify with `chemsplit.audit.audit_split`.
> - Encourages leaderboard over-fitting, since the community tunes against one fixed test set for years.

In [10]:
n = 6
assignment = ["train", "train", "test", "test", "valid", "discard"]
sp = PredefinedSplitter(assignment=assignment)
[result] = sp.split_result(np.arange(n, dtype=np.float64).reshape(-1, 1))
summarize(result)
print("train:", result.train.tolist(), " test:", result.test.tolist(),
      " valid:", result.valid.tolist(), " discard:", result.discard.tolist())

fold_column = [0, 0, 1, 1, -1, -1]
sp2 = PredefinedSplitter(fold_column=fold_column)
fold_results = sp2.split_result(np.arange(n, dtype=np.float64).reshape(-1, 1))
print(f"\n{len(fold_results)} folds from fold_column")
for i, r in enumerate(fold_results):
    print(f"  fold {i}:", end=" ")
    summarize(r)

train=2 valid=1 test=2 discard=1
train: [0, 1]  test: [2, 3]  valid: [4]  discard: [5]

2 folds from fold_column
  fold 0: train=4 valid=0 test=2 discard=0
  fold 1: train=4 valid=0 test=2 discard=0


Both input forms — an explicit partition-name list, and a numeric fold column — reproduce
the assignment exactly, byte for byte.

---

That's the full `baseline` family: five zero-chemistry controls that every structural splitter
in the other eight families should be benchmarked against. See the `scaffold`, `similarity`, and
`biomolecular` notebooks for the chemistry-aware alternatives these are meant to be compared to.